# Pipeline 1 - Donor Lapse Risk Predictor

Generated: 2026-04-07T11:53:52.836463Z


## 1) Problem Framing

**Business question:** Identify supporters most likely to lapse in the next 90 days so leaders can prioritize retention outreach.

**Who cares:** nonprofit leadership, program staff, and/or fundraising staff depending on the pipeline domain.

**Why it matters:** this model turns operational, donor, or outreach data into a decision-support signal for a resource-constrained safehouse nonprofit.

**Predictive vs. explanatory goal:** this notebook includes both. The predictive model is evaluated on unseen data and is used for deployment-oriented scoring. The explanatory or relationship model is included to identify which variables appear most connected to the target and to support business interpretation. We do not treat predictive accuracy as causal proof.

**Success metrics:** classification pipelines use accuracy, F1, and ROC AUC where appropriate. Regression/forecasting pipelines use MAE, RMSE, and R-squared. The notebook also compares against a simple baseline so the results can be interpreted honestly.


## Notebook Setup

Shared imports and helper functions are defined once here so the later rubric sections can focus on the pipeline-specific code for this business problem.


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from data_prep import (
    as_bool,
    compare_profiles,
    dataset_profile,
    export_predictions_json,
    fill_numeric_median,
    get_project_paths,
    load_df,
    numeric,
    prep,
    quick_eda,
    safe_classifier_split,
    score_bands,
)
from modeling import (
    CandidateResult,
    cv_evaluate_candidate,
    cv_results_table,
    eval_classification,
    read_json_if_exists,
    select_simplest_within_delta_cv,
    should_export_by_guardrail,
    top_features,
    tune_model_randomized,
    write_ablation_report_json,
    write_run_report_json,
    ablate_feature_groups_one_by_one_cv,
)

paths = get_project_paths()
print("Data dir:", paths.data_dir)
print("Output dir:", paths.out_dir)


def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)


## 2) Data Acquisition, Preparation, and Exploration

The pipeline reads the provided CSV files from `data/raw/`, performs joins and feature engineering in code, handles missing values reproducibly, and prints an EDA summary before modeling. The EDA step checks row counts, missingness, target distributions, feature summaries, and key categorical distributions.


In [ ]:
supporters = load_df("supporters")
donations = load_df("donations")
donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
donations = donations.dropna(subset=["donation_date", "supporter_id"]).copy()
donations["amount"] = numeric(donations["amount"])
donations["estimated_value"] = numeric(donations["estimated_value"])
donations["is_recurring_bool"] = as_bool(donations["is_recurring"])

cutoff = donations["donation_date"].max() - pd.Timedelta(days=90)
label_end = cutoff + pd.Timedelta(days=90)
past = donations[donations["donation_date"] <= cutoff].copy()
future = donations[(donations["donation_date"] > cutoff) & (donations["donation_date"] <= label_end)].copy()

base = supporters[["supporter_id","supporter_type","relationship_type","region","country","status","acquisition_channel","first_donation_date"]].copy()
base["first_donation_date"] = pd.to_datetime(base["first_donation_date"], errors="coerce")
agg = past.groupby("supporter_id").agg(
    donation_count=("donation_id","count"),
    last_donation_date=("donation_date","max"),
    distinct_campaigns=("campaign_name", lambda s: s.dropna().nunique()),
    distinct_channels=("channel_source", lambda s: s.dropna().nunique()),
    recurring_any=("is_recurring_bool","max"),
    total_value=("estimated_value","sum"),
).reset_index()
mon = past[past["donation_type"] == "Monetary"]
mon_agg = mon.groupby("supporter_id").agg(
    monetary_count=("donation_id","count"),
    monetary_sum=("amount","sum"),
    monetary_avg=("amount","mean"),
    monetary_max=("amount","max"),
).reset_index()
future_gave = (future.groupby("supporter_id")["donation_id"].count() > 0).astype(int).rename("gave_next_90d").reset_index()

df = base.merge(agg, on="supporter_id", how="left").merge(mon_agg, on="supporter_id", how="left").merge(future_gave, on="supporter_id", how="left")
df = df[df["donation_count"].fillna(0) > 0].copy()
df["gave_next_90d"] = df["gave_next_90d"].fillna(0).astype(int)
df["lapsed_next_90d"] = 1 - df["gave_next_90d"]
df["recency_days"] = (cutoff - df["last_donation_date"]).dt.days.clip(lower=0)
df["donor_age_days"] = (cutoff - df["first_donation_date"]).dt.days.clip(lower=0)
cat_cols = ["supporter_type","relationship_type","region","country","status","acquisition_channel"]
num_cols = ["donation_count","distinct_campaigns","distinct_channels","recurring_any","total_value","monetary_count","monetary_sum","monetary_avg","monetary_max","recency_days","donor_age_days"]
df[cat_cols] = df[cat_cols].fillna("Unknown")
fill_numeric_median(df, num_cols)
print("Rows:", len(df), "lapse rate:", round(df["lapsed_next_90d"].mean(), 3))
quick_eda(df, "Donor lapse modeling table", target_col="lapsed_next_90d", numeric_cols=num_cols, categorical_cols=cat_cols)


## 3) Modeling and Feature Selection

The feature set is selected from fields that would be available at the decision point whenever possible. Predictive models use ensembles or tuned tree-based models when they improve out-of-sample performance. Explanatory models use simpler linear or logistic models when interpretability matters.


In [ ]:
features = cat_cols + num_cols
X_train, X_test, y_train, y_test = safe_classifier_split(df[features], df["lapsed_next_90d"])
predictive = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", GradientBoostingClassifier(random_state=42))])
explanatory = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", LogisticRegression(max_iter=3000))])


## 4) Evaluation and Selection

The notebook uses a train/test or time-based holdout split depending on the business problem. Where appropriate, it also uses compact cross-validation, model comparison tables, and lightweight randomized tuning. Metrics are interpreted in business terms rather than treated as abstract statistics.


In [ ]:
# Strict holdout discipline: CV on train, single touch on holdout.
X_train_full, X_holdout, y_train_full, y_holdout = safe_classifier_split(df[features], df["lapsed_next_90d"])

report_path = paths.reports_dir / "donor-lapse-risk_run_report.json"
prev_report = read_json_if_exists(report_path)

current_profile = dataset_profile(df[features], categorical_cols=cat_cols, numeric_cols=num_cols)
drift = compare_profiles(current_profile, (prev_report or {}).get("data_profile"))

candidates = {
    "LogisticRegression": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", LogisticRegression(max_iter=3000))]),
    "RandomForestSmall": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestClassifier(n_estimators=200, random_state=42, min_samples_leaf=3))]),
    "GradientBoosting": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", GradientBoostingClassifier(random_state=42))]),
}

# 1) CV tournament
cv_results = [cv_evaluate_candidate(name, est, X_train_full, y_train_full, task="binary", top_fraction=0.10) for name, est in candidates.items()]
cv_tbl = cv_results_table(cv_results).sort_values(by=["recall_at_top10pct_mean", "pr_auc_mean"], ascending=False)
print("CV tournament (train only):")
print(cv_tbl.to_string(index=False))

selected_cv = select_simplest_within_delta_cv(cv_results, primary_metric="recall_at_top10pct", delta=0.02, higher_is_better=True)
base_selected = selected_cv.estimator
base_selected_name = selected_cv.name

# 2) Tune selected family
param_space = {}
if base_selected_name.startswith("RandomForest"):
    param_space = {"model__n_estimators": [100, 150, 200, 300], "model__max_depth": [None, 3, 5, 8], "model__min_samples_leaf": [1, 3, 5, 8]}
elif base_selected_name.startswith("GradientBoosting"):
    param_space = {"model__n_estimators": [100, 150, 250], "model__learning_rate": [0.03, 0.05, 0.1], "model__max_depth": [2, 3, 4]}
elif base_selected_name.startswith("Logistic"):
    param_space = {"model__C": [0.25, 0.5, 1.0, 2.0, 4.0]}

if param_space:
    tune = tune_model_randomized(
        base_selected_name,
        base_selected,
        X_train_full,
        y_train_full,
        param_distributions=param_space,
        task="binary",
        cv_metric="average_precision",
        n_iter=12,
    )
    tuned_estimator = tune.best_estimator
    tuned_name = f"{base_selected_name}+tuned"
    tuned_params = tune.best_params
else:
    tuned_estimator = base_selected
    tuned_name = base_selected_name
    tuned_params = {}

# 3) Post-tune CV check
post_cv = cv_evaluate_candidate(tuned_name, tuned_estimator, X_train_full, y_train_full, task="binary", top_fraction=0.10)

# 4) Ablation report
ablation_tbl = ablate_feature_groups_one_by_one_cv(
    base_name=tuned_name,
    estimator=tuned_estimator,
    X=X_train_full,
    y=y_train_full,
    task="binary",
    feature_groups=[[c] for c in features],
    primary_metric="recall_at_top10pct",
    higher_is_better=True,
    top_fraction=0.10,
)
write_ablation_report_json(paths.reports_dir / "donor-lapse-risk_ablation_report.json", {"pipeline": "donor-lapse-risk", "table": ablation_tbl.to_dict(orient="records")})

# 5) Final holdout report
final_model = tuned_estimator
final_model.fit(X_train_full, y_train_full)
holdout_proba = final_model.predict_proba(X_holdout)[:, 1]
holdout_pred = (holdout_proba >= 0.5).astype(int)
holdout_metrics = eval_classification(y_holdout, holdout_pred, holdout_proba)

allow_export, guardrail = should_export_by_guardrail(
    previous_report=prev_report,
    current_holdout={"pr_auc": float(holdout_metrics.get("pr_auc", 0.0))},
    primary_metric="pr_auc",
    min_delta_allowed=0.01,
    higher_is_better=True,
)

write_run_report_json(
    report_path,
    {
        "pipeline": "donor-lapse-risk",
        "primary_metric": "recall_at_top10pct",
        "selected_family": base_selected_name,
        "tuned_name": tuned_name,
        "tuned_params": tuned_params,
        "cv_table": cv_tbl.to_dict(orient="records"),
        "post_tune_cv": post_cv.metrics_mean,
        "holdout": holdout_metrics,
        "guardrail": guardrail,
        "data_profile": current_profile,
        "drift": drift,
        "notes": {"review_capacity_top_fraction": 0.10},
    },
)
print("Wrote run report:", report_path)

if not allow_export:
    print("Export blocked by guardrail:", guardrail)

# Keep explanatory model for interpretation
explanatory.fit(X_train_full, y_train_full)
predictive = final_model


## 5) Causal and Relationship Analysis

The relationship analysis section highlights important features and discusses whether those relationships make sense for the organization. These findings are observational: they can guide hypotheses and strategy, but they are not automatically causal. Any sensitive resident-care decision must remain human-reviewed.


In [ ]:
print("Top predictive features:")
print(top_features(predictive).to_string(index=False))
print("Top explanatory relationships:")
print(top_features(explanatory).to_string(index=False))
print_business_takeaway("Prioritize high-risk supporters for retention outreach, especially when recency and low engagement patterns suggest they may lapse.")


## 6) Deployment Notes

The final scoring step exports JSON to `output/ml-predictions/`. These files match the API import contract used by `POST /api/ml/import?replace=true` and can be viewed in the deployed staff portal under `/app/ml` or the ML action center.


In [ ]:
df_out = df[["supporter_id"] + features].copy()
df_out["risk_score"] = predictive.predict_proba(df_out[features])[:, 1]
df_out["risk_band"] = score_bands(df_out["risk_score"])
if allow_export:
    export_predictions_json(
        "donor_lapse_90d",
        "Supporter",
        df_out[["supporter_id","risk_score","risk_band","recency_days","donation_count","monetary_sum","recurring_any","acquisition_channel","supporter_type"]].assign(export_model=tuned_name),
        "supporter_id",
        "risk_score",
        "risk_band",
    )
else:
    print("Skipping export due to guardrail.")
